In [ ]:
import tensorflow as tf
import pickle
import numpy as np
import re
import os

from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
DATA_DIR = "../data/processed"
MODEL_DIR = "../models"

MODEL_PATH = os.path.join(MODEL_DIR,"joint_model_final.keras")
TOKENIZER_PATH = os.path.join(DATA_DIR,"tokenizer.pkl")

In [ ]:

# ── Text Cleaning ──────────────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    """Remove HTML, URLs, emails, special chars from real job posting text."""
    if not isinstance(text, str):
        text = str(text) if text is not None else ""
    text = text.lower()
    text = re.sub(r"<[^>]+>",               " ", text)   # HTML tags
    text = re.sub(r"&[a-z]+;",              " ", text)   # HTML entities
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)   # URLs
    text = re.sub(r"\S+@\S+",               " ", text)   # emails
    text = re.sub(r"[^a-z0-9\s]",           " ", text)   # special chars
    text = re.sub(r"\s+",                   " ", text).strip()
    return text


In [ ]:

# ── Custom Maxout Layer ────────────────────────────────────────────────────────
class MaxoutLayer(layers.Layer):
    """
    Maxout activation (Goodfellow et al., 2013).
    Each output unit takes the max over `num_pieces` linear projections.
    Piecewise-linear activation — expressive and works well with Dropout.
    """
    def __init__(self, units: int, num_pieces: int = 2, l2: float = 1e-4, **kwargs):
        super().__init__(**kwargs)
        self.units      = units
        self.num_pieces = num_pieces
        self.l2_val     = l2

    def build(self, input_shape):
        input_dim = int(input_shape[-1])
        reg = regularizers.l2(self.l2_val)
        self.W = self.add_weight(
            name="W", shape=(input_dim, self.units * self.num_pieces),
            initializer="glorot_uniform", regularizer=reg, trainable=True)
        self.b = self.add_weight(
            name="b", shape=(self.units * self.num_pieces,),
            initializer="zeros", trainable=True)
        super().build(input_shape)

    def call(self, inputs):
        z = tf.matmul(inputs, self.W) + self.b
        z = tf.reshape(z, (-1, self.units, self.num_pieces))
        return tf.reduce_max(z, axis=-1)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"units": self.units, "num_pieces": self.num_pieces, "l2": self.l2_val})
        return cfg


In [ ]:
with open(TOKENIZER_PATH,"rb") as f:
    tokenizer = pickle.load(f)

In [ ]:
model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={"MaxoutLayer": MaxoutLayer}
)

In [4]:
import os, re, pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [5]:
MODEL_DIR = "../models"
DATA_DIR = "../data/processed"

JOINT_MODEL_PATH = os.path.join(MODEL_DIR, "joint_model_final.keras")
TOKENIZER_PATH = os.path.join(DATA_DIR, "tokenizer.pkl")

MAX_SEQUENCE_LENGTH = 256
THRESHOLD = 0.4

In [6]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"&[a-z]+;", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [7]:
class MaxoutLayer(layers.Layer):

    def __init__(self, units, num_pieces=2, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.num_pieces = num_pieces

    def build(self, input_shape):

        input_dim = int(input_shape[-1])

        self.W = self.add_weight(
            shape=(input_dim, self.units * self.num_pieces),
            initializer="glorot_uniform",
            trainable=True
        )

        self.b = self.add_weight(
            shape=(self.units * self.num_pieces,),
            initializer="zeros",
            trainable=True
        )

    def call(self, inputs):

        z = tf.matmul(inputs, self.W) + self.b
        z = tf.reshape(z, (-1, self.units, self.num_pieces))

        return tf.reduce_max(z, axis=-1)

In [8]:
print("Loading tokenizer...")

with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)

print("Loading model...")

model = load_model(
    JOINT_MODEL_PATH,
    custom_objects={"MaxoutLayer": MaxoutLayer}
)

print("Model loaded successfully")

Loading tokenizer...
Loading model...


TypeError: <class 'keras.src.models.functional.Functional'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.src.models.functional', 'class_name': 'Functional', 'config': {}, 'registered_name': 'Functional', 'build_config': {'input_shape': None}, 'compile_config': {'optimizer': {'module': 'keras.optimizers', 'class_name': 'Adam', 'config': {'name': 'adam', 'learning_rate': 0.0005000000237487257, 'weight_decay': None, 'clipnorm': 1.0, 'global_clipnorm': None, 'clipvalue': None, 'use_ema': False, 'ema_momentum': 0.99, 'ema_overwrite_frequency': None, 'loss_scale_factor': None, 'gradient_accumulation_steps': None, 'beta_1': 0.9, 'beta_2': 0.999, 'epsilon': 1e-07, 'amsgrad': False}, 'registered_name': None}, 'loss': {'module': 'keras.losses', 'class_name': 'BinaryCrossentropy', 'config': {'name': 'bi

In [ ]:
def predict_job(text):

    cleaned = clean_text(text)

    seq = tokenizer.texts_to_sequences([cleaned])

    padded = pad_sequences(
        seq,
        maxlen=MAX_SEQUENCE_LENGTH,
        padding="post",
        truncating="post"
    )

    prob = float(model.predict(padded, verbose=0)[0][0])

    label = "Fake Job" if prob > THRESHOLD else "Real Job"

    return label, prob

In [ ]:
text = """
Work from home data entry job.
Earn 500 dollars weekly.
No experience required.
Small registration fee required.
"""

predict_job(text)